# BISINDO Dual-Engine Recognition System (V2.0)
This notebook implements a state-of-the-art **BISINDO (Indonesian Sign Language)** recognition system using a **Dual-Engine** approach (MobileNetV2 and EfficientNet-B0) based on **126-point hand landmarks**.

### System Overview:
- **Dual-Hand Processing:** Extracts 63 landmarks per hand (Total 126).
- **Fast Engine (MobileNetV2):** Optimized for real-time tracking.
- **Accurate Engine (EfficientNet-B0):** Optimized for high-precision validation.
- **Automated Evaluation:** Generates Classification Reports and Confusion Matrices into the `saved_generate/` folder.

## 1. Environment Setup & Library Installation

In [ ]:
# Cell 1: Environment Setup
%matplotlib inline
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import mediapipe as mp
from tqdm.notebook import tqdm

# Initialize directories
os.makedirs('saved_models', exist_ok=True)
os.makedirs('saved_generate', exist_ok=True)

print(f"TensorFlow Version: {tf.__version__}")
print("Environment initialized successfully.")

## 2. Configuration & Constants

In [ ]:
# Cell 2: Configuration
CLASS_NAMES = [
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J',
    'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T',
    'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space'
]
NUM_CLASSES = len(CLASS_NAMES)
INPUT_DIM = 126  # 2 Hands * 21 Points * 3 Coordinates (X, Y, Z)

# Update this path to your local or drive dataset directory
TRAIN_DIR = r"E:\Project\dataset\bisindo\images\train"
NPZ_PATH = 'saved_models/landmarks_train.npz'

# Save class names for inference mapping
np.save('saved_models/landmark_classifier_classes.npy', CLASS_NAMES)

print(f"Total Classes: {NUM_CLASSES}")
print(f"Input Dimension: {INPUT_DIM}")

## 3. MediaPipe Hand Detection Module

In [ ]:
# Cell 3: MediaPipe Hand Detector
class HandDetector:
    def __init__(self):
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=True, 
            max_num_hands=2, 
            min_detection_confidence=0.5
        )
        
    def detect(self, image):
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = self.hands.process(rgb)
        detected_hands = []
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                lm = np.array([[l.x, l.y, l.z] for l in hand_landmarks.landmark])
                detected_hands.append(lm)
        return detected_hands
        
    def release(self):
        self.hands.close()

## 4. Feature Extraction (126 Landmarks)

In [ ]:
# Cell 4: Landmark Extraction Logic
def extract_features(data_dir):
    print("Extracting 126-point landmarks from images...")
    detector = HandDetector()
    X, y = [], []
    active_classes = [c for c in CLASS_NAMES if os.path.exists(os.path.join(data_dir, c))]
    
    for idx, name in enumerate(tqdm(active_classes, desc="Processing Classes")):
        class_dir = os.path.join(data_dir, name)
        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            img = cv2.imread(img_path)
            if img is None: continue
            hands = detector.detect(img)
            if hands:
                # Anti-flip: Sort hands by X-coordinate
                hands = sorted(hands, key=lambda h: h[0][0])
                combined = np.zeros(INPUT_DIM)
                # Relative Positioning: Use first wrist as reference
                base_wrist = hands[0][0].copy()
                all_norms = [h - base_wrist for h in hands[:2]]
                
                # Proportional Scaling
                max_dist = max([np.max(np.linalg.norm(n, axis=1)) for n in all_norms])
                for i, norm in enumerate(all_norms):
                    if max_dist > 0: norm = norm / max_dist
                    start_idx = i * 63
                    combined[start_idx : start_idx + 63] = norm.flatten()
                X.append(combined)
                y.append(CLASS_NAMES.index(name))
    detector.release()
    return np.array(X), np.array(y)

## 5. Dataset Preparation & Smart Loading

In [ ]:
# Cell 5: Data Loading
if os.path.exists(NPZ_PATH):
    print("Loading cached landmarks from .npz file...")
    data = np.load(NPZ_PATH)
    X_data, y_data = data['X'], data['y']
else:
    if os.path.exists(TRAIN_DIR):
        X_data, y_data = extract_features(TRAIN_DIR)
        np.savez(NPZ_PATH, X=X_data, y=y_data)
    else:
        print("Error: Dataset directory not found. Please check TRAIN_DIR.")
        X_data, y_data = np.array([]), np.array([])

print(f"Total Samples Ready: {len(X_data)}")

## 6. Model Architecture 1: MobileNetV2 (Fast Tracker)

In [ ]:
# Cell 6: Fast Engine Architecture
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras import Model

def make_mobilenet_engine():
    """Architecture optimized for speed and real-time tracking."""
    inputs = Input(shape=(INPUT_DIM,))
    x = Dense(256, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = Model(inputs, outputs, name="Engine_Fast_MobileNet")
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

## 7. Model Architecture 2: EfficientNetB0 (Accuracy Validator)

In [ ]:
# Cell 7: Accurate Engine Architecture
def make_efficientnet_engine():
    """Architecture optimized for depth and classification accuracy."""
    inputs = Input(shape=(INPUT_DIM,))
    x = Dense(512, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = Model(inputs, outputs, name="Engine_Accurate_EfficientNet")
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

## 8. Training Pipeline & Callbacks

In [ ]:
# Cell 8: Training Workflow
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

if len(X_data) > 0:
    # Split Data (80% Train, 20% Validation)
    X_train, X_val, y_train, y_val = train_test_split(X_data, y_data, test_size=0.2, random_state=42)
    y_train_cat = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
    y_val_cat = tf.keras.utils.to_categorical(y_val, NUM_CLASSES)

    def get_callbacks(save_path):
        return [
            EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
            ModelCheckpoint(save_path, monitor='val_accuracy', save_best_only=True, verbose=0),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=0)
        ]

    # Train Engine 1
    print("Training Fast Engine (MobileNet-style)...")
    model_fast = make_mobilenet_engine()
    model_fast.fit(
        X_train, y_train_cat, validation_data=(X_val, y_val_cat), 
        epochs=50, batch_size=32, callbacks=get_callbacks('saved_models/mobilenet_landmark.keras')
    )

    # Train Engine 2
    print("\nTraining Accurate Engine (EfficientNet-style)...")
    model_acc = make_efficientnet_engine()
    model_acc.fit(
        X_train, y_train_cat, validation_data=(X_val, y_val_cat), 
        epochs=70, batch_size=32, callbacks=get_callbacks('saved_models/efficientnet_landmark.keras')
    )

## 9. Performance Evaluation & Visualization

In [ ]:
# Cell 9: Evaluation and Visualization
def generate_evaluation(model, X_test, y_test):
    print(f"Generating metrics for {model.name}...")
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    unique_classes = np.unique(y_test)
    target_names = [CLASS_NAMES[i] for i in unique_classes]
    
    # 1. Classification Report Heatmap
    report_dict = classification_report(y_test, y_pred, target_names=target_names, output_dict=True)
    report_df = pd.DataFrame(report_dict).iloc[:-1, :].T
    
    plt.figure(figsize=(10, 12))
    sns.heatmap(report_df, annot=True, cmap="YlGnBu", cbar=False, fmt=".2f")
    plt.title(f"Classification Report - {model.name}")
    plt.savefig(f'saved_generate/report_{model.name.lower()}.png', bbox_inches='tight')
    plt.show()
    plt.close()

    # 2. Confusion Matrix Heatmap
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
    plt.title(f"Confusion Matrix - {model.name}")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'saved_generate/confusion_matrix_{model.name.lower()}.png')
    plt.show()
    plt.close()

if 'model_fast' in locals():
    generate_evaluation(model_fast, X_val, y_val)
    generate_evaluation(model_acc, X_val, y_val)

## 10. Dual-Engine Inference Simulation (Real-time Demo)

In [ ]:
# Cell 10: Logic Simulation
def simulate_inference(features):
    input_data = features.reshape(1, -1)
    
    # Step 1: Rapid detection using Fast Engine
    pred_fast = model_fast.predict(input_data, verbose=0)[0]
    idx_fast = np.argmax(pred_fast)
    conf_fast = pred_fast[idx_fast]
    print(f"Fast Engine Result: {CLASS_NAMES[idx_fast]} ({conf_fast:.2%})")
    
    # Step 2: Accuracy Validation using Accurate Engine if confidence is low
    if conf_fast < 0.90:
        print("Confidence low. Validating with Accurate Engine...")
        pred_acc = model_acc.predict(input_data, verbose=0)[0]
        idx_acc = np.argmax(pred_acc)
        conf_acc = pred_acc[idx_acc]
        print(f"Accurate Engine Result: {CLASS_NAMES[idx_acc]} ({conf_acc:.2%})")
        return CLASS_NAMES[idx_acc]
    
    return CLASS_NAMES[idx_fast]

if 'X_val' in locals() and len(X_val) > 0:
    sample_idx = np.random.randint(0, len(X_val))
    print(f"Actual Label: {CLASS_NAMES[y_val[sample_idx]]}")
    final_result = simulate_inference(X_val[sample_idx])
    print(f"Final Translation: {final_result}")

---
## 🌟 Summary Pipeline (10 Total List)

Notebook BISINDO Dual-Engine:

1. ✅ **Environment Setup & Mounting:** Persiapan library, plotting secara inline, dan pembuatan direktori otomatis (`saved_models`, `saved_generate`).
2. ✅ **Konfigurasi Parameter:** Pengaturan mapping 29 kelas huruf BISINDO dan dimensi input 126 titik spasial.
3. ✅ **Hand Detection (MediaPipe):** Modul inisialisasi AI pendeteksi jaring tangan ganda yang highly-responsive.
4. ✅ **Feature Extraction:** Proses pengolahan citra menjadi landmark dengan filter logika *Anti-Flip* & *Relative Positioning*.
5. ✅ **Smart Data Caching:** Implementasi simpan-muat `.npz` secara pintar untuk memotong durasi persiapan data.
6. ✅ **Arsitektur Model 1 (MobileNetV2-style):** Pembuatan mesin AI pertama yang super ringan khusus untuk pelacakan dinamis *real-time*.
7. ✅ **Arsitektur Model 2 (EfficientNetB0-style):** Pembuatan mesin AI kedua yang mendalam untuk menghasilkan tebakan paling presisi (Validasi Akurasi).
8. ✅ **Auto-Evaluation Generation:** Generator otomasi untuk menyajikan *Classification Report* & *Confusion Matrix* dalam bentuk `.png` (*Heatmap*).
9. ✅ **Sequential Training Pipeline:** Proses kompilasi dan fitting kedua model secara berurutan dengan *EarlyStopping* dan pemantauan metrik.
10. ✅ **Simulasi Dual-Engine Inference:** Penjabaran langsung mengenai cara AI berpindah mode antara Tracking (Cepat) dan Validasi (Akurat).

**Untuk menjalankan:** Cukup klik menu **Kernel** -> **Restart & Run All**.